In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
#!/usr/bin/env python3
# ============================================================
# Con φ  ~  USE  (Update Survey Estimate)
# ============================================================
#
# VERSION BLOCK
# -------------
# Set these two variables once here. All paths and output
# filenames are derived from them automatically.
# If you are running this notebook standalone (not via the
# master runner), uncomment the lines below and set your own
# BASE_DIR.
#
VERSION = "v1"
BASE_DIR_DEFAULT = "/content/drive/MyDrive/conphi"
#
# ── Per-script override (uncomment if running standalone) ────
# from pathlib import Path
# VERSION  = "v1"
# BASE_DIR = Path("/content/drive/MyDrive/conphi")   # 👈 change to your path
# ─────────────────────────────────────────────────────────────
#
# If BASE_DIR is not already defined (i.e. running standalone
# without the master runner), fall back to the default above.
try:
    BASE_DIR
except NameError:
    from pathlib import Path
    BASE_DIR = Path(BASE_DIR_DEFAULT)

BASE_DIR = Path(BASE_DIR)

# ============================================================
# Con φ — USE  (Update Survey Estimate)
# ============================================================
"""
Con φ USE — Update Survey Estimate
====================================

USE is one of two sub-models in the Con φ system used by the
World Food Programme for global expenditure monitoring.  It is
used when a country has an existing consumption survey: USE
projects that survey distribution forward to the present using
observed and forecast GDP growth.

CORE IDEA
---------
If we know a country's consumption distribution at some past
survey date, and we know what happened to GDP between then and
now, we can project what the consumption distribution looks
like today.  USE learns the "passthrough rate" — how much of
GDP growth reaches households — from historical pairs of
comparable consumption surveys.

WHY ASYMMETRIC PASSTHROUGH?
----------------------------
Economic contractions do not simply reverse the gains of
expansions.  During downturns, consumption tends to fall more
sharply than it rises during booms.  USE therefore estimates
separate passthrough rates for positive and negative GDP growth
years, allowing recessions and expansions to have different
effects on household consumption.

MODEL SPECIFICATION
-------------------
For consecutive surveys within the same comparable spell:

    Δlog(cons_p) = β⁺(p) × Σ⁺ Δlog(GDP)
                 + β⁻(p) × Σ⁻ Δlog(GDP)  + ε

    β⁺(p) = β₀⁺ + β_p⁺ × logit(p)
    β⁻(p) = β₀⁻ + β_p⁻ × logit(p)

    ε ~ StudentT(ν, 0, σ)

Annual GDP growth between the two survey years is decomposed
year-by-year into positive and negative components:

    Σ⁺ = Σ_{t=y₀+1}^{y₁}  max(g_t, 0)     (expansion years)
    Σ⁻ = Σ_{t=y₀+1}^{y₁}  min(g_t, 0)     (contraction years)

PARAMETERS
----------
    β₀⁺   — average passthrough in expansion years
    β₀⁻   — average passthrough in contraction years
    β_p⁺  — how passthrough varies across the distribution
             in expansions
    β_p⁻  — how passthrough varies across the distribution
             in contractions
    σ     — observation noise scale
    ν     — degrees of freedom controlling tail heaviness

The logit(p) term lets passthrough vary by percentile.
A negative β_p means poorer households benefit proportionally
more from growth (pro-poor); a positive value means richer
households gain more.

LIKELIHOOD
----------
A Student-t likelihood provides heavier tails than a Normal,
yielding well-calibrated 90% predictive intervals.  The degrees
of freedom ν are estimated from the data — small ν (≈3–5)
indicates heavy tails; large ν recovers the Normal.

ESTIMATION
----------
Stochastic Variational Inference (SVI) with a Block Neural
Autoregressive Flow (BNAF) guide, implemented in NumPyro/JAX.

VINTAGE CONTROL
---------------
Each run is anchored to a specific vintage year.  Consumption
outcomes at or after the vintage year are masked before
training, so the model only learns from data that would have
been available at that point in time.  GDP growth beyond the
vintage year comes from IMF WEO forecasts embedded in the
corresponding feature file.

INPUTS
------
    conphi/data - model inputs/conphi_v1_features_{year}.parquet

OUTPUTS
-------
    conphi/outputs/conphi_v1_use_{mode_suffix}/
      ├── target_year={year}/
      │   ├── conphi_v1_use_posterior_samples_{year}.npz
      │   ├── conphi_v1_use_inference_data_{year}.nc
      │   ├── conphi_v1_use_posterior_summary_{year}.csv
      │   ├── conphi_v1_use_train_pairs_{year}.parquet
      │   ├── conphi_v1_use_logit_p_spec_{year}.json
      │   ├── conphi_v1_use_predictions_{mode_suffix}_{year}.parquet
      │   └── conphi_v1_use_metrics_{mode_suffix}_{year}.parquet
      ├── conphi_v1_use_predictions_master_{mode_suffix}.parquet
      ├── conphi_v1_use_metrics_master_{mode_suffix}.parquet
      ├── conphi_v1_use_metrics_by_year_horizon_{mode_suffix}.parquet
      ├── conphi_v1_use_metrics_by_horizon_{mode_suffix}.parquet
      ├── conphi_v1_use_metrics_by_year_{mode_suffix}.parquet
      ├── conphi_v1_use_metrics_overall_{mode_suffix}.parquet
      ├── conphi_v1_use_run_metadata_{mode_suffix}.json
      └── conphi_v1_use_output_summary_{mode_suffix}.json
"""

# ============================================================
# BLOCK 1 — IMPORTS & GPU SETUP
# ============================================================
import subprocess
import sys
import json
import gc
import time
import shutil
import warnings
import platform
from pathlib import Path

warnings.filterwarnings("ignore")


def install_if_needed(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", pkg, "-q", "--break-system-packages"]
        )


install_if_needed("numpyro")
install_if_needed("jax[cuda12]", "jax")
install_if_needed("arviz")
install_if_needed("optax")

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import arviz as az
import numpyro
import numpyro.distributions as dist
import optax

from jax import random as jrandom
from numpyro.infer import SVI, Trace_ELBO, Predictive
from numpyro.infer.autoguide import AutoBNAFNormal
from scipy.special import logit as sp_logit
from scipy.stats import t as scipy_t

print(f"JAX devices : {jax.devices()}")
print(f"JAX backend : {jax.default_backend()}")
USE_GPU = jax.default_backend() == "gpu"
print(f"Using GPU   : {USE_GPU}")

numpyro.set_platform("cuda" if USE_GPU else "cpu")
numpyro.set_host_device_count(1)


# ============================================================
# BLOCK 2 — CONFIGURATION
# ============================================================
MODEL_NAME    = "conphi_v1_use"
MODEL_VERSION = VERSION

# ── I/O directories ───────────────────────────────────────────
INPUT_DIR = BASE_DIR / "data - model inputs"

# ── Prediction mode ───────────────────────────────────────────
# "nowcast"  — project to the vintage year itself
# "forecast" — project FORECAST_HORIZON years beyond the vintage
PREDICTION_MODE  = "forecast"   # "nowcast" or "forecast"
FORECAST_HORIZON = 1            # 1–5 years ahead (forecast mode only)

assert PREDICTION_MODE in ("nowcast", "forecast")
assert 1 <= FORECAST_HORIZON <= 5

if PREDICTION_MODE == "nowcast":
    MODE_SUFFIX = "nowcast"
else:
    MODE_SUFFIX = f"forecast_{FORECAST_HORIZON}yr"

SAVE_DIR = BASE_DIR / "outputs" / f"{MODEL_NAME}_{MODE_SUFFIX}"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ── Flush previous outputs ────────────────────────────────────
# Set True to wipe the output directory before running.
# Ensures no stale checkpoint files are reused after code changes.
FLUSH_OUTPUTS = True

# ── Maximum survey gap (dt) ───────────────────────────────────
# Training pairs spanning more than MAX_DT years are dropped.
# Very long gaps accumulate forecast errors and structural noise
# that make the GDP-consumption relationship unreliable.
# Set to None to disable.
MAX_DT = 50

ALLOW_PURE_PREDICTION = True

# ── Vintage years to process ──────────────────────────────────
YEARS = list(range(2015, 2026))

# ── Column names ──────────────────────────────────────────────
TARGET_COL = "log_cons"
GROWTH_COL = "gdp_growth"

# ── ISOs to exclude ───────────────────────────────────────────
DROP_ISOS = ["XKX"]

# ── SVI hyperparameters ───────────────────────────────────────
SVI_STEPS               = 10_000
SVI_LR                  = 0.005
POSTERIOR_SAMPLES       = 1_000
POSTERIOR_SAMPLE_GROUPS = 4
N_PRED_DRAWS            = 500
RANDOM_SEED             = 42

# ── Likelihood ────────────────────────────────────────────────
LIKELIHOOD_NAME = "StudentT"

# ── Posterior parameter names ─────────────────────────────────
POSTERIOR_VAR_NAMES = [
    "beta0_pos", "beta0_neg",
    "beta_p_pos", "beta_p_neg",
    "sigma", "nu",
]


# ============================================================
# BLOCK 3 — HELPERS
# ============================================================
def flush_output_directory(save_dir: Path):
    """Remove and recreate the output directory."""
    if save_dir.exists():
        n_files = sum(1 for _ in save_dir.rglob("*") if _.is_file())
        print(f"  FLUSH: removing {save_dir} ({n_files} files)")
        shutil.rmtree(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    print(f"  FLUSH: clean directory created at {save_dir}")


def year_dir(target_year: int) -> Path:
    out = SAVE_DIR / f"target_year={target_year}"
    out.mkdir(parents=True, exist_ok=True)
    return out


def to_numpy_dict(samples):
    return {k: np.asarray(v) for k, v in samples.items()}


def save_numpy_samples(samples: dict, path: Path):
    np.savez_compressed(path, **to_numpy_dict(samples))


def json_ready(obj):
    if isinstance(obj, dict):
        return {k: json_ready(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_ready(v) for v in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(json_ready(obj), f, indent=2)


def check_duplicate_percentiles(df, label):
    dupes = df.duplicated(subset=["iso", "year", "percentile"], keep=False)
    if dupes.any():
        n = int(dupes.sum())
        print(f"  WARNING: {label} has {n} duplicated iso-year-percentile rows. Keeping first.")
        df = df.drop_duplicates(subset=["iso", "year", "percentile"], keep="first").copy()
    return df


def cumulative_growth_asymmetric(gdp_dict, iso, y_start, y_end):
    """
    Walk through each year from y_start+1 to y_end and split
    annual GDP growth into its positive and negative components.

    Returns (cum_g_pos, cum_g_neg, valid_flag).
    valid_flag is False if any year in the range is missing GDP.
    """
    cum_g_pos = 0.0
    cum_g_neg = 0.0
    for y in range(y_start + 1, y_end + 1):
        g = gdp_dict.get((iso, y), np.nan)
        if not np.isfinite(g):
            return np.nan, np.nan, False
        if g > 0:
            cum_g_pos += g
        else:
            cum_g_neg += g
    return cum_g_pos, cum_g_neg, True


def standardize_logit_p(logit_p_raw):
    """
    Standardize logit(percentile) to mean 0, sd 1.
    Returns (standardized_values, center, scale).
    """
    vals   = np.asarray(logit_p_raw, dtype=float)
    finite = vals[np.isfinite(vals)]
    center = float(np.mean(finite))
    scale  = float(np.std(finite))
    if not np.isfinite(scale) or scale < 1e-8:
        scale = 1.0
    return (vals - center) / scale, center, scale


class SVIWrapper:
    """Thin wrapper giving SVI results a .get_samples() API."""
    def __init__(self, samples, params):
        self.samples = samples
        self.params  = params

    def get_samples(self, group_by_chain=False):
        return self.samples


def summarize_metrics(metrics_df, group_cols):
    if metrics_df is None or len(metrics_df) == 0:
        return pd.DataFrame()
    num_cols = [
        "mae_cons", "rmse_cons", "mape_pct",
        "mae_log", "rmse_log", "bias_log", "r2_log",
        "coverage_90_pct",
    ]
    agg_dict = {"n_eval_rows": ("n_eval_rows", "sum")}
    for c in num_cols:
        if c in metrics_df.columns:
            agg_dict[f"mean_{c}"] = (c, "mean")
    return (
        metrics_df
        .groupby(group_cols, dropna=False)
        .agg(**agg_dict)
        .reset_index()
    )


def write_master_outputs(all_pred_frames, all_metrics_frames, elapsed_minutes=None):
    """Write consolidated master output files across all vintage years."""
    sfx   = MODE_SUFFIX
    preds = pd.concat(all_pred_frames,   ignore_index=True) if all_pred_frames   else pd.DataFrame()
    mets  = pd.concat(all_metrics_frames, ignore_index=True) if all_metrics_frames else pd.DataFrame()

    if len(preds) > 0:
        preds.to_parquet(
            SAVE_DIR / f"{MODEL_NAME}_predictions_master_{sfx}.parquet", index=False
        )
    if len(mets) > 0:
        mets.to_parquet(
            SAVE_DIR / f"{MODEL_NAME}_metrics_master_{sfx}.parquet", index=False
        )
        summarize_metrics(mets, ["target_year", "horizon"]).to_parquet(
            SAVE_DIR / f"{MODEL_NAME}_metrics_by_year_horizon_{sfx}.parquet", index=False
        )
        summarize_metrics(mets, ["horizon"]).to_parquet(
            SAVE_DIR / f"{MODEL_NAME}_metrics_by_horizon_{sfx}.parquet", index=False
        )
        summarize_metrics(mets, ["target_year"]).to_parquet(
            SAVE_DIR / f"{MODEL_NAME}_metrics_by_year_{sfx}.parquet", index=False
        )
        summarize_metrics(mets, ["model_name"]).to_parquet(
            SAVE_DIR / f"{MODEL_NAME}_metrics_overall_{sfx}.parquet", index=False
        )

    save_json(
        {
            "model_name":       MODEL_NAME,
            "model_version":    MODEL_VERSION,
            "prediction_mode":  PREDICTION_MODE,
            "forecast_horizon": FORECAST_HORIZON if PREDICTION_MODE == "forecast" else 0,
            "mode_suffix":      MODE_SUFFIX,
            "prediction_rows":  int(len(preds)),
            "metric_rows":      int(len(mets)),
            "save_dir":         str(SAVE_DIR),
            "elapsed_minutes":  elapsed_minutes,
        },
        SAVE_DIR / f"{MODEL_NAME}_output_summary_{sfx}.json",
    )


# ============================================================
# BLOCK 4 — DATA LOADING
# ============================================================
def load_and_prep(vintage_year):
    """
    Load a single Con φ feature file and prepare it for USE
    training and prediction.

    Each feature file is a vintage-controlled snapshot of the
    world as it was known at the start of that year: all
    consumption surveys published before that date, plus GDP
    estimates and forecasts from the corresponding IMF WEO
    release.  Consumption outcomes at or after the vintage year
    are masked so the model never trains on data it would not
    have had in real time.

    Returns:
      df       — consumption survey rows (outcome masked for
                 year >= vintage_year)
      gdp_dict — {(iso, year): growth_rate} for all country-years
    """
    fp = INPUT_DIR / f"conphi_{VERSION}_features_{vintage_year}.parquet"
    if not fp.exists():
        raise FileNotFoundError(fp)

    df_full = pd.read_parquet(fp)

    # ── GDP lookup — built from ALL rows before filtering ─────
    # Needed for every country-year (including those without
    # surveys) so that prediction can project forward from any
    # anchor year.
    _gdp = (
        df_full[["iso", "year", GROWTH_COL]]
        .drop_duplicates(["iso", "year"])
        .copy()
    )
    _gdp["iso"]         = _gdp["iso"].astype(str).str.strip()
    _gdp["year"]        = pd.to_numeric(_gdp["year"], errors="coerce")
    _gdp[GROWTH_COL]    = pd.to_numeric(_gdp[GROWTH_COL], errors="coerce")
    _gdp               = _gdp.dropna(subset=["iso", "year", GROWTH_COL]).copy()
    _gdp["year"]        = _gdp["year"].astype(int)

    if DROP_ISOS:
        _gdp = _gdp[~_gdp["iso"].isin(DROP_ISOS)].copy()

    gdp_dict   = {(r.iso, r.year): r[GROWTH_COL] for _, r in _gdp.iterrows()}
    n_gdp_isos = _gdp["iso"].nunique()

    # ── Filter to consumption surveys ─────────────────────────
    df = df_full[df_full["welfare_type"] == "consumption"].copy()

    if DROP_ISOS:
        df = df[~df["iso"].isin(DROP_ISOS)].copy()

    df["year"]       = pd.to_numeric(df["year"],       errors="coerce").astype("Int64")
    df["percentile"] = pd.to_numeric(df["percentile"], errors="coerce").astype("Int64")
    df["cons"]       = pd.to_numeric(df["cons"],       errors="coerce").astype(float)

    # ── Outcome masking ───────────────────────────────────────
    future_mask = df["year"] >= vintage_year
    n_masked    = int(future_mask.sum())
    for col in ["cons", "log_cons"]:
        if col in df.columns:
            df.loc[future_mask, col] = np.nan
    print(f"  Outcome masking: {n_masked:,} rows at year >= {vintage_year} set to NaN")

    if "log_cons" not in df.columns:
        df["log_cons"] = np.nan
    df["log_cons"] = np.where(df["cons"] > 0, np.log(df["cons"]), np.nan)

    p            = pd.to_numeric(df["percentile"], errors="coerce").astype(float) / 100.0
    p            = np.clip(p, 1e-6, 1 - 1e-6)
    df["logit_p"] = sp_logit(p)

    if "comparable_spell_raw" not in df.columns:
        raise ValueError("Missing required column: comparable_spell_raw")
    df["comparable_spell_raw"] = df["comparable_spell_raw"].where(
        df["comparable_spell_raw"].notna(), None
    )

    df = df.dropna(subset=["iso", "year", "percentile"]).copy()
    df["year"]       = df["year"].astype(int)
    df["percentile"] = df["percentile"].astype(int)
    df = check_duplicate_percentiles(df, label=f"load_and_prep({vintage_year})")

    yr_range         = f"{df['year'].min()}–{df['year'].max()}"
    n_with_outcome   = int(df["log_cons"].notna().sum())
    print(
        f"  Consumption rows: {len(df):,}, {df['iso'].nunique()} ISOs, "
        f"years {yr_range}, {n_with_outcome:,} with outcome"
    )
    print(f"  GDP lookup: {len(gdp_dict):,} entries across {n_gdp_isos} ISOs")

    return df, gdp_dict


# ============================================================
# BLOCK 5 — TRAINING PAIR CONSTRUCTION
#
# Each training observation is a pair of consumption surveys
# from the same country within the same comparable spell — a
# period during which the underlying survey methodology is
# consistent enough that changes in measured consumption
# genuinely reflect changes in living standards.
#
# For each pair we record:
#   - the change in log consumption at every percentile
#   - the cumulative positive and negative GDP growth between
#     the two survey years (year-by-year decomposition)
#
# The MAX_DT filter drops pairs where the survey gap is too
# large, as very long spans make the passthrough relationship
# unreliable.
# ============================================================
def build_training_pairs(df, gdp_dict, target_year):
    """Build consumption-GDP passthrough training pairs."""
    train = df[(df["year"] < target_year) & df["log_cons"].notna()].copy()
    if train.empty:
        print(f"  No outcome rows with year < {target_year}")
        return pd.DataFrame()

    iso_year_spell = (
        train.loc[
            train["comparable_spell_raw"].notna(),
            ["iso", "year", "comparable_spell_raw"],
        ]
        .drop_duplicates(["iso", "year"])
        .sort_values(["iso", "comparable_spell_raw", "year"])
    )

    pairs = []
    for (iso, spell), grp in iso_year_spell.groupby(
        ["iso", "comparable_spell_raw"], dropna=True
    ):
        yrs = grp["year"].sort_values().tolist()
        for i in range(len(yrs) - 1):
            y0, y1 = yrs[i], yrs[i + 1]
            dt     = y1 - y0
            if dt <= 0:
                continue
            if MAX_DT is not None and dt >= MAX_DT:
                continue
            cum_g_pos, cum_g_neg, valid = cumulative_growth_asymmetric(
                gdp_dict, iso, y0, y1
            )
            if not valid:
                continue
            pairs.append({
                "iso":               iso,
                "spell":             spell,
                "y_start":           y0,
                "y_end":             y1,
                "dt":                dt,
                "delta_log_gdp_pos": cum_g_pos,
                "delta_log_gdp_neg": cum_g_neg,
                "delta_log_gdp":     cum_g_pos + cum_g_neg,
            })

    if not pairs:
        print(f"  No valid training pairs for target {target_year}")
        return pd.DataFrame()

    pairs_df = pd.DataFrame(pairs)
    if MAX_DT is not None:
        print(f"  dt cutoff: excluding pairs with dt >= {MAX_DT} years")

    # Expand to percentile level
    pct_cols = ["iso", "year", "percentile", "log_cons", "logit_p"]
    pct_data = (
        train[pct_cols]
        .dropna(subset=["log_cons"])
        .drop_duplicates(["iso", "year", "percentile"])
        .copy()
    )
    start = pct_data.rename(columns={
        "year": "y_start", "log_cons": "log_cons_start", "logit_p": "logit_p_",
    })
    end = pct_data.rename(columns={
        "year": "y_end", "log_cons": "log_cons_end",
    })

    out = pairs_df.merge(
        start[["iso", "y_start", "percentile", "log_cons_start", "logit_p_"]],
        on=["iso", "y_start"], how="inner",
    )
    out = out.merge(
        end[["iso", "y_end", "percentile", "log_cons_end"]],
        on=["iso", "y_end", "percentile"], how="inner",
    )

    out["delta_log_cons"] = out["log_cons_end"] - out["log_cons_start"]
    out["logit_p"]        = out["logit_p_"]
    out = out.drop(columns=["logit_p_"]).dropna(
        subset=["delta_log_cons", "delta_log_gdp_pos", "delta_log_gdp_neg", "logit_p"]
    )

    n_mixed     = int(((out["delta_log_gdp_pos"] > 0) & (out["delta_log_gdp_neg"] < 0)).sum())
    n_pure_pos  = int((out["delta_log_gdp_neg"] == 0).sum())
    n_pure_neg  = int((out["delta_log_gdp_pos"] == 0).sum())
    dt_range    = f"{int(out['dt'].min())}–{int(out['dt'].max())}"
    print(
        f"  Training pairs: {len(pairs_df)} survey pairs → "
        f"{len(out):,} obs ({out['iso'].nunique()} ISOs), dt range: {dt_range} yrs"
    )
    print(
        f"    GDP decomposition: {n_pure_pos:,} pure expansion, "
        f"{n_pure_neg:,} pure contraction, {n_mixed:,} mixed"
    )
    return out


# ============================================================
# BLOCK 6 — NUMPYRO MODEL
#
# Δlog(cons_p) = (β₀⁺ + β_p⁺ × logit_p_std) × g_pos
#              + (β₀⁻ + β_p⁻ × logit_p_std) × g_neg  + ε
#
# ε ~ StudentT(ν, 0, σ)
#
# Priors:
#   β₀⁺   ~ Normal(0.70, 0.30)  expansion passthrough
#   β₀⁻   ~ Normal(0.70, 0.50)  contraction passthrough (wider —
#                                 fewer contraction episodes in data)
#   β_p⁺  ~ Normal(0.00, 0.10)  expansion distributional tilt
#   β_p⁻  ~ Normal(0.00, 0.30)  contraction distributional tilt
#   σ     ~ HalfNormal(0.10)    observation noise scale
#   ν     ~ Gamma(2, 0.1)       degrees of freedom (mean ≈ 20)
# ============================================================
def consumption_growth_model(g_pos, g_neg, logit_p_std, y=None):
    """
    Con φ USE — asymmetric Student-t passthrough model.

    Parameters
    ----------
    g_pos        : cumulative positive GDP growth  (Σ max(g_t, 0))
    g_neg        : cumulative negative GDP growth  (Σ min(g_t, 0))
    logit_p_std  : standardized logit(percentile), mean 0, sd 1
    y            : observed Δlog(cons); None during prediction
    """
    n = g_pos.shape[0]

    # Expansion parameters
    beta0_pos  = numpyro.sample("beta0_pos",  dist.Normal(0.70, 0.30))
    beta_p_pos = numpyro.sample("beta_p_pos", dist.Normal(0.00, 0.10))

    # Contraction parameters (wider priors — sparse contraction data)
    beta0_neg  = numpyro.sample("beta0_neg",  dist.Normal(0.70, 0.50))
    beta_p_neg = numpyro.sample("beta_p_neg", dist.Normal(0.00, 0.30))

    # Noise parameters
    sigma = numpyro.sample("sigma", dist.HalfNormal(0.10))
    nu    = numpyro.sample("nu",    dist.Gamma(2.0, 0.1))

    # Percentile-varying passthrough
    beta_total_pos = beta0_pos + beta_p_pos * logit_p_std
    beta_total_neg = beta0_neg + beta_p_neg * logit_p_std

    mu = beta_total_pos * g_pos + beta_total_neg * g_neg
    numpyro.deterministic("mu_pred", mu)

    with numpyro.plate("obs", n):
        numpyro.sample("y_obs", dist.StudentT(nu, mu, sigma), obs=y)


# ============================================================
# BLOCK 7 — SVI FITTING
# ============================================================
def prepare_train_data(train_df, logit_p_center, logit_p_scale):
    """Prepare JAX arrays for SVI fitting."""
    g_pos        = train_df["delta_log_gdp_pos"].values.astype(np.float32)
    g_neg        = train_df["delta_log_gdp_neg"].values.astype(np.float32)
    logit_p_raw  = train_df["logit_p"].values.astype(np.float32)
    logit_p_std  = ((logit_p_raw - logit_p_center) / logit_p_scale).astype(np.float32)

    return {
        "g_pos":       jnp.array(g_pos),
        "g_neg":       jnp.array(g_neg),
        "logit_p_std": jnp.array(logit_p_std),
        "y":           jnp.array(train_df["delta_log_cons"].values, dtype=jnp.float32),
    }


def fit_svi(train_data, rng_key):
    """Fit USE via SVI with a BNAF normalizing-flow guide."""
    guide     = AutoBNAFNormal(
        consumption_growth_model, num_flows=1, hidden_factors=[8, 8],
    )
    optimizer = optax.adam(learning_rate=SVI_LR)
    svi       = SVI(consumption_growth_model, guide, optimizer, loss=Trace_ELBO())

    train_kwargs = {
        "g_pos":       train_data["g_pos"],
        "g_neg":       train_data["g_neg"],
        "logit_p_std": train_data["logit_p_std"],
    }

    svi_result = svi.run(
        rng_key, num_steps=SVI_STEPS, progress_bar=False,
        **train_kwargs, y=train_data["y"],
    )

    rng_key, key_guide, key_model = jrandom.split(rng_key, 3)
    n_samples = POSTERIOR_SAMPLES * POSTERIOR_SAMPLE_GROUPS

    guide_predictive  = Predictive(guide, params=svi_result.params, num_samples=n_samples)
    guide_samples     = guide_predictive(key_guide)

    model_predictive  = Predictive(consumption_growth_model, posterior_samples=guide_samples)
    model_samples     = model_predictive(key_model, **train_kwargs, y=None)

    all_samples = {**guide_samples, **model_samples}
    return SVIWrapper(all_samples, svi_result.params)


# ============================================================
# BLOCK 8 — PREDICTION
#
# For each country with a usable survey anchor, USE projects
# consumption forward by applying the learned passthrough rates
# to cumulative GDP growth between the anchor year and the
# prediction year.  Predictive uncertainty comes from two
# sources: parameter uncertainty (posterior draws giving
# different passthrough rates) and observation noise
# (Student-t draws with per-sample ν and σ).
# ============================================================
def predict_multiple_years(
    df, gdp_dict, target_year, prediction_years,
    posterior_samples, logit_p_center, logit_p_scale,
):
    """Generate USE predictions for one or more years."""
    prior = df[(df["year"] < target_year) & df["log_cons"].notna()].copy()
    if prior.empty:
        return pd.DataFrame()

    # Best anchor per iso: most recent survey within a comparable spell
    anchor_candidates = (
        prior[prior["comparable_spell_raw"].notna()]
        .groupby(["iso", "comparable_spell_raw"])["year"]
        .max()
        .reset_index()
        .rename(columns={"year": "anchor_year"})
    )
    best_anchor = (
        anchor_candidates
        .sort_values("anchor_year", ascending=False)
        .drop_duplicates("iso", keep="first")
    )

    geo_cols = [c for c in ["region", "region23"] if c in df.columns]
    if geo_cols:
        iso_geo = df[["iso"] + geo_cols].drop_duplicates("iso").set_index("iso")
    else:
        iso_geo = None

    # Subsample posterior draws for efficiency
    post     = to_numpy_dict(posterior_samples)
    rng      = np.random.default_rng(RANDOM_SEED)
    n_draws  = post["beta0_pos"].shape[0]
    draw_idx = rng.choice(n_draws, size=min(N_PRED_DRAWS, n_draws), replace=False)
    n_used   = len(draw_idx)

    beta0_pos_draws  = post["beta0_pos"][draw_idx]
    beta0_neg_draws  = post["beta0_neg"][draw_idx]
    beta_p_pos_draws = post["beta_p_pos"][draw_idx]
    beta_p_neg_draws = post["beta_p_neg"][draw_idx]
    sigma_draws      = post["sigma"][draw_idx]
    nu_draws         = post["nu"][draw_idx]

    all_preds = []

    for _, arow in best_anchor.iterrows():
        iso         = arow["iso"]
        anchor_year = int(arow["anchor_year"])

        anchor_data = (
            df[(df["iso"] == iso) & (df["year"] == anchor_year)]
            [["percentile", "log_cons", "logit_p"]]
            .dropna(subset=["log_cons"])
            .drop_duplicates("percentile")
            .sort_values("percentile")
        )
        if anchor_data.empty:
            continue

        pcts            = anchor_data["percentile"].to_numpy()
        log_cons_anchor = anchor_data["log_cons"].to_numpy()        # (P,)
        logit_p_raw     = anchor_data["logit_p"].to_numpy()
        logit_p_std     = (logit_p_raw - logit_p_center) / logit_p_scale  # (P,)

        geo = {}
        if iso_geo is not None and iso in iso_geo.index:
            for c in geo_cols:
                geo[c] = iso_geo.loc[iso, c]

        for pred_year in prediction_years:
            horizon = pred_year - target_year
            dt      = pred_year - anchor_year
            if dt <= 0:
                continue

            cum_g_pos, cum_g_neg, valid = cumulative_growth_asymmetric(
                gdp_dict, iso, anchor_year, pred_year
            )
            if not valid:
                continue

            # Percentile-varying passthrough: (n_used, P)
            beta_total_pos = (
                beta0_pos_draws[:, None]
                + beta_p_pos_draws[:, None] * logit_p_std[None, :]
            )
            beta_total_neg = (
                beta0_neg_draws[:, None]
                + beta_p_neg_draws[:, None] * logit_p_std[None, :]
            )

            # Predicted consumption growth: (n_used, P)
            delta_draws = beta_total_pos * cum_g_pos + beta_total_neg * cum_g_neg

            # Latent predictions: (n_used, P)
            log_cons_pred_latent = log_cons_anchor[None, :] + delta_draws

            # Observation noise: Student-t draw per posterior sample
            n_pcts = len(pcts)
            noise  = np.zeros((n_used, n_pcts))
            for i in range(n_used):
                noise[i, :] = scipy_t.rvs(
                    df=nu_draws[i], loc=0.0, scale=sigma_draws[i],
                    size=n_pcts, random_state=rng,
                )
            log_cons_pred_obs = log_cons_pred_latent + noise

            # Posterior summaries
            mean_pred = log_cons_pred_latent.mean(axis=0)
            std_pred  = log_cons_pred_latent.std(axis=0)
            lo5_mu    = np.percentile(log_cons_pred_latent, 5,  axis=0)
            hi95_mu   = np.percentile(log_cons_pred_latent, 95, axis=0)
            lo5_obs   = np.percentile(log_cons_pred_obs,    5,  axis=0)
            hi95_obs  = np.percentile(log_cons_pred_obs,    95, axis=0)

            for j, pctl in enumerate(pcts):
                row = {
                    "model_name":        MODEL_NAME,
                    "prediction_mode":   PREDICTION_MODE,
                    "mode_suffix":       MODE_SUFFIX,
                    "target_year":       int(target_year),
                    "horizon":           int(horizon),
                    "iso":               iso,
                    "year":              int(pred_year),
                    "percentile":        int(pctl),
                    "anchor_year":       int(anchor_year),
                    "dt":                int(dt),
                    "delta_log_gdp_pos": float(cum_g_pos),
                    "delta_log_gdp_neg": float(cum_g_neg),
                    "delta_log_gdp":     float(cum_g_pos + cum_g_neg),
                    "log_cons_hat":      float(mean_pred[j]),
                    "log_cons_hat_std":  float(std_pred[j]),
                    "mu_q05":            float(lo5_mu[j]),
                    "mu_q95":            float(hi95_mu[j]),
                    "y_q05":             float(lo5_obs[j]),
                    "y_q95":             float(hi95_obs[j]),
                    "cons_hat":          float(np.exp(mean_pred[j])),
                    "cons_hat_q05":      float(np.exp(lo5_mu[j])),
                    "cons_hat_q95":      float(np.exp(hi95_mu[j])),
                }
                for c in geo_cols:
                    row[c] = geo.get(c, None)
                all_preds.append(row)

    n_isos = len(set(r["iso"] for r in all_preds)) if all_preds else 0
    print(f"  Predictions: {len(all_preds):,} rows across {n_isos} ISOs")
    return pd.DataFrame(all_preds)


# ============================================================
# BLOCK 9 — EVALUATION
# ============================================================
def evaluate_predictions_horizon(df_preds_h, df_actuals, target_year, horizon):
    out = {
        "model_name":       MODEL_NAME,
        "prediction_mode":  PREDICTION_MODE,
        "mode_suffix":      MODE_SUFFIX,
        "target_year":      int(target_year),
        "horizon":          int(horizon),
        "n_pred_rows":      int(len(df_preds_h)),
        "n_eval_rows":      0,
        "n_countries_eval": 0,
        "mae_cons":         np.nan,
        "rmse_cons":        np.nan,
        "mape_pct":         np.nan,
        "mae_log":          np.nan,
        "rmse_log":         np.nan,
        "bias_log":         np.nan,
        "r2_log":           np.nan,
        "coverage_90_pct":  np.nan,
    }

    if df_preds_h.empty or df_actuals.empty:
        return pd.DataFrame([out])

    pred_year  = target_year + horizon
    actuals_yr = df_actuals[df_actuals["year"] == pred_year].copy()
    if actuals_yr.empty:
        return pd.DataFrame([out])

    ev = df_preds_h.merge(
        actuals_yr[["iso", "year", "percentile", "log_cons", "cons"]].rename(
            columns={"log_cons": "log_cons_actual", "cons": "cons_actual"}
        ),
        on=["iso", "year", "percentile"],
        how="inner",
    )
    ev = ev.dropna(subset=["log_cons_actual", "cons_actual"])
    if ev.empty:
        return pd.DataFrame([out])

    c_true  = ev["cons_actual"].values
    c_hat   = ev["cons_hat"].values
    c_resid = c_hat - c_true

    with np.errstate(divide="ignore", invalid="ignore"):
        ape = np.abs(c_resid / c_true)
        ape = np.where(np.isfinite(ape), ape, np.nan)

    y_true  = ev["log_cons_actual"].values
    y_hat   = ev["log_cons_hat"].values
    l_resid = y_hat - y_true
    ss_res  = np.sum(l_resid ** 2)
    ss_tot  = np.sum((y_true - y_true.mean()) ** 2)

    coverage_90 = np.mean(
        (y_true >= ev["y_q05"].values) & (y_true <= ev["y_q95"].values)
    ) * 100

    out.update({
        "n_eval_rows":      int(len(ev)),
        "n_countries_eval": int(ev["iso"].nunique()),
        "mae_cons":         float(np.mean(np.abs(c_resid))),
        "rmse_cons":        float(np.sqrt(np.mean(c_resid ** 2))),
        "mape_pct":         float(np.nanmean(ape) * 100),
        "mae_log":          float(np.mean(np.abs(l_resid))),
        "rmse_log":         float(np.sqrt(np.mean(l_resid ** 2))),
        "bias_log":         float(l_resid.mean()),
        "r2_log":           float(1.0 - ss_res / ss_tot) if ss_tot > 1e-8 else np.nan,
        "coverage_90_pct":  float(coverage_90),
    })
    return pd.DataFrame([out])


def merge_actuals_into_predictions(df_preds, df_actuals):
    if df_preds.empty:
        return df_preds
    actual_cols = ["iso", "year", "percentile", "log_cons", "cons"]
    actual_cols = [c for c in actual_cols if c in df_actuals.columns]
    act         = df_actuals[actual_cols].drop_duplicates(["iso", "year", "percentile"]).copy()
    rename_map  = {}
    if "log_cons" in act.columns: rename_map["log_cons"] = "log_cons_actual"
    if "cons"     in act.columns: rename_map["cons"]     = "cons_actual"
    act = act.rename(columns=rename_map)
    return df_preds.merge(act, on=["iso", "year", "percentile"], how="left")


# ============================================================
# BLOCK 10 — SAVE / LOAD
# ============================================================
def save_model_artifacts(target_year, posterior_fit, train_df, logit_p_spec):
    out_dir           = year_dir(target_year)
    posterior_samples = posterior_fit.get_samples(group_by_chain=False)

    save_numpy_samples(
        posterior_samples,
        out_dir / f"{MODEL_NAME}_posterior_samples_{target_year}.npz",
    )
    save_json(
        logit_p_spec,
        out_dir / f"{MODEL_NAME}_logit_p_spec_{target_year}.json",
    )

    try:
        post_for_az = {
            k: np.asarray(posterior_samples[k])
            for k in POSTERIOR_VAR_NAMES
            if k in posterior_samples
            and np.all(np.isfinite(np.asarray(posterior_samples[k])))
        }
        if post_for_az:
            idata   = az.from_dict(posterior=post_for_az)
            az.to_netcdf(idata, out_dir / f"{MODEL_NAME}_inference_data_{target_year}.nc")
            summary = az.summary(idata, var_names=list(post_for_az.keys()), round_to=6)
            summary.to_csv(out_dir / f"{MODEL_NAME}_posterior_summary_{target_year}.csv")
            print(f"  Posterior summary:")
            for idx, row in summary.iterrows():
                print(
                    f"    {idx:>12s}: mean={row['mean']:.4f}  "
                    f"sd={row['sd']:.4f}  "
                    f"[{row['hdi_3%']:.4f}, {row['hdi_97%']:.4f}]"
                )
    except Exception as e:
        print(f"  Warning: ArviZ summary error: {e}")

    train_df.to_parquet(
        out_dir / f"{MODEL_NAME}_train_pairs_{target_year}.parquet", index=False
    )


def save_prediction_artifacts(target_year, df_preds, metrics_df):
    out_dir = year_dir(target_year)
    sfx     = MODE_SUFFIX
    df_preds.to_parquet(
        out_dir / f"{MODEL_NAME}_predictions_{sfx}_{target_year}.parquet", index=False
    )
    metrics_df.to_parquet(
        out_dir / f"{MODEL_NAME}_metrics_{sfx}_{target_year}.parquet", index=False
    )


def build_run_metadata():
    return {
        "model_name":             MODEL_NAME,
        "model_version":          MODEL_VERSION,
        "likelihood":             LIKELIHOOD_NAME,
        "prediction_mode":        PREDICTION_MODE,
        "forecast_horizon":       FORECAST_HORIZON if PREDICTION_MODE == "forecast" else 0,
        "mode_suffix":            MODE_SUFFIX,
        "allow_pure_prediction":  ALLOW_PURE_PREDICTION,
        "max_dt":                 MAX_DT,
        "flush_outputs":          FLUSH_OUTPUTS,
        "target_col":             TARGET_COL,
        "growth_col":             GROWTH_COL,
        "growth_definition":      "diff(log_gdp_pp) — first difference of log PPP GDP per capita",
        "asymmetric_passthrough": True,
        "decomposition":          "year-by-year: g_pos = Σ max(g_t, 0), g_neg = Σ min(g_t, 0)",
        "seed":                   RANDOM_SEED,
        "svi_steps":              SVI_STEPS,
        "svi_lr":                 SVI_LR,
        "posterior_samples":      POSTERIOR_SAMPLES * POSTERIOR_SAMPLE_GROUPS,
        "n_pred_draws":           N_PRED_DRAWS,
        "drop_isos":              DROP_ISOS,
        "years":                  YEARS,
        "base_dir":               str(BASE_DIR),
        "input_dir":              str(INPUT_DIR),
        "save_dir":               str(SAVE_DIR),
        "python_version":         sys.version,
        "platform":               platform.platform(),
        "jax_backend":            jax.default_backend(),
        "devices":                [str(d) for d in jax.devices()],
        "numpy_version":          np.__version__,
        "pandas_version":         pd.__version__,
        "jax_version":            jax.__version__,
        "numpyro_version":        numpyro.__version__,
        "arviz_version":          az.__version__,
    }


# ============================================================
# BLOCK 11 — MAIN LOOP
# ============================================================
def main():
    grand_t0   = time.time()
    master_key = jrandom.PRNGKey(RANDOM_SEED)
    sfx        = MODE_SUFFIX

    print("=" * 80)
    print(f"Con φ {VERSION}  ~  USE  (Update Survey Estimate)")
    print(f"  Model : Δlog(cons) = β⁺(p)×Σ⁺Δlog(GDP) + β⁻(p)×Σ⁻Δlog(GDP) + ε")
    print(f"  β⁺(p) = β₀⁺ + β_p⁺ × logit(p)   [expansion passthrough]")
    print(f"  β⁻(p) = β₀⁻ + β_p⁻ × logit(p)   [contraction passthrough]")
    print(f"  ε     ~ StudentT(ν, 0, σ)")
    print(f"  GDP decomposition: year-by-year (Σ⁺ = Σ max(g_t,0), Σ⁻ = Σ min(g_t,0))")
    print(f"  Prediction mode : {PREDICTION_MODE}")
    if PREDICTION_MODE == "forecast":
        print(f"  Forecast horizon: {FORECAST_HORIZON} year(s) ahead")
    print(f"  Mode suffix     : {sfx}")
    print(f"  Max survey gap  : {MAX_DT if MAX_DT is not None else 'no limit'} years")
    print(f"  Flush outputs   : {FLUSH_OUTPUTS}")
    print(f"  Input dir       : {INPUT_DIR}")
    print(f"  Output dir      : {SAVE_DIR}")
    print("=" * 80)

    if FLUSH_OUTPUTS:
        flush_output_directory(SAVE_DIR)

    save_json(build_run_metadata(), SAVE_DIR / f"{MODEL_NAME}_run_metadata_{sfx}.json")

    all_pred_frames    = []
    all_metrics_frames = []

    for target_year in YEARS:
        yr_t0      = time.time()
        master_key, key_fit = jrandom.split(master_key)
        out_dir    = year_dir(target_year)

        # ── Checkpoint: skip if results already exist ─────────
        pred_path    = out_dir / f"{MODEL_NAME}_predictions_{sfx}_{target_year}.parquet"
        metrics_path = out_dir / f"{MODEL_NAME}_metrics_{sfx}_{target_year}.parquet"
        if pred_path.exists() and metrics_path.exists():
            print(f"\n[{target_year}] CHECKPOINT: loading existing results")
            all_pred_frames.append(pd.read_parquet(pred_path))
            all_metrics_frames.append(pd.read_parquet(metrics_path))
            continue

        print(f"\n[{target_year}] TARGET YEAR")

        # ── Load vintage feature file ─────────────────────────
        try:
            df, gdp_dict = load_and_prep(target_year)
        except FileNotFoundError:
            print(f"  Skipping {target_year}: feature file not found")
            continue

        # ── Build training pairs ──────────────────────────────
        train_df = build_training_pairs(df, gdp_dict, target_year=target_year)
        if train_df.empty:
            continue

        # ── Standardize logit(p) from training data ───────────
        logit_p_std_vals, logit_p_center, logit_p_scale = standardize_logit_p(
            train_df["logit_p"].values
        )
        train_df["logit_p_std"] = logit_p_std_vals
        logit_p_spec = {"center": logit_p_center, "scale": logit_p_scale}

        # ── Fit ───────────────────────────────────────────────
        train_data   = prepare_train_data(train_df, logit_p_center, logit_p_scale)
        posterior_fit = fit_svi(train_data, key_fit)
        save_model_artifacts(target_year, posterior_fit, train_df, logit_p_spec)

        # ── Determine prediction years ────────────────────────
        if PREDICTION_MODE == "nowcast":
            prediction_years = [target_year]
        else:
            prediction_years = list(range(target_year, target_year + FORECAST_HORIZON + 1))

        # ── Predict ───────────────────────────────────────────
        df_preds = predict_multiple_years(
            df, gdp_dict, target_year, prediction_years,
            posterior_fit.get_samples(), logit_p_center, logit_p_scale,
        )

        if df_preds.empty:
            print(f"  No predictions generated for {target_year}")
            del posterior_fit; gc.collect()
            continue

        # ── Load actuals for evaluation ───────────────────────
        df_full_actuals = pd.read_parquet(
            INPUT_DIR / f"conphi_{VERSION}_features_{target_year}.parquet"
        )
        df_actuals = df_full_actuals[
            df_full_actuals["welfare_type"] == "consumption"
        ].copy()
        if DROP_ISOS:
            df_actuals = df_actuals[~df_actuals["iso"].isin(DROP_ISOS)].copy()
        df_actuals = df_actuals.dropna(subset=["log_cons"]).drop_duplicates(
            ["iso", "year", "percentile"]
        )

        # ── Evaluate per horizon ──────────────────────────────
        fold_metrics_frames = []
        has_any_eval        = False
        for h in sorted(df_preds["horizon"].unique()):
            df_h      = df_preds[df_preds["horizon"] == h].copy()
            metrics_h = evaluate_predictions_horizon(
                df_h, df_actuals, target_year, horizon=int(h)
            )
            fold_metrics_frames.append(metrics_h)
            if int(metrics_h["n_eval_rows"].iloc[0]) > 0:
                has_any_eval = True

        metrics_df = pd.concat(fold_metrics_frames, ignore_index=True)
        df_preds   = merge_actuals_into_predictions(df_preds, df_actuals)

        if not has_any_eval and not ALLOW_PURE_PREDICTION:
            print(f"  No evaluable rows and ALLOW_PURE_PREDICTION=False. Skipping.")
            del posterior_fit; gc.collect()
            continue

        # ── Save ──────────────────────────────────────────────
        save_prediction_artifacts(target_year, df_preds, metrics_df)
        all_pred_frames.append(df_preds.copy())
        all_metrics_frames.append(metrics_df.copy())

        elapsed = time.time() - yr_t0

        # ── Print per-horizon summary ─────────────────────────
        for _, mrow in metrics_df.iterrows():
            h        = int(mrow["horizon"])
            n_eval   = int(mrow["n_eval_rows"])
            n_ctries = int(mrow["n_countries_eval"])
            if n_eval > 0:
                print(
                    f"  [h={h}] {n_ctries} countries, {n_eval} rows | "
                    f"MAE=${mrow['mae_cons']:.3f}/day | "
                    f"RMSE=${mrow['rmse_cons']:.3f}/day | "
                    f"MAPE={mrow['mape_pct']:.1f}% | "
                    f"R²={mrow['r2_log']:.4f} | "
                    f"Cov90={mrow['coverage_90_pct']:.1f}%"
                )
            else:
                print(
                    f"  [h={h}] {int(mrow['n_pred_rows'])} predictions, "
                    f"no actuals for evaluation"
                )

        print(f"  Done in {elapsed:.1f}s")
        del posterior_fit; gc.collect()

        # ── Incremental master save ───────────────────────────
        write_master_outputs(
            all_pred_frames, all_metrics_frames,
            elapsed_minutes=round((time.time() - grand_t0) / 60, 2),
        )

    # ── Final master outputs ──────────────────────────────────
    print("\n" + "=" * 80)
    print(f"WRITING FINAL MASTER OUTPUTS  ({sfx.upper()})")
    print("=" * 80)

    write_master_outputs(
        all_pred_frames, all_metrics_frames,
        elapsed_minutes=round((time.time() - grand_t0) / 60, 2),
    )

    total_mins = (time.time() - grand_t0) / 60
    print(f"\nCon φ {VERSION}  USE  complete.")
    print(f"Mode   : {PREDICTION_MODE}"
          + (f"  |  horizon: {FORECAST_HORIZON} yr" if PREDICTION_MODE == "forecast" else ""))
    print(f"Outputs: {SAVE_DIR}")
    print(f"Runtime: {total_mins:.1f} minutes")


if __name__ == "__main__":
    main()

JAX devices : [CudaDevice(id=0)]
JAX backend : gpu
Using GPU   : True
Con φ v1  ~  USE  (Update Survey Estimate)
  Model : Δlog(cons) = β⁺(p)×Σ⁺Δlog(GDP) + β⁻(p)×Σ⁻Δlog(GDP) + ε
  β⁺(p) = β₀⁺ + β_p⁺ × logit(p)   [expansion passthrough]
  β⁻(p) = β₀⁻ + β_p⁻ × logit(p)   [contraction passthrough]
  ε     ~ StudentT(ν, 0, σ)
  GDP decomposition: year-by-year (Σ⁺ = Σ max(g_t,0), Σ⁻ = Σ min(g_t,0))
  Prediction mode : forecast
  Forecast horizon: 1 year(s) ahead
  Mode suffix     : forecast_1yr
  Max survey gap  : 50 years
  Flush outputs   : True
  Input dir       : /content/drive/MyDrive/conphi/data - model inputs
  Output dir      : /content/drive/MyDrive/conphi/outputs/conphi_v1_use_forecast_1yr
  FLUSH: removing /content/drive/MyDrive/conphi/outputs/conphi_v1_use_forecast_1yr (0 files)
  FLUSH: clean directory created at /content/drive/MyDrive/conphi/outputs/conphi_v1_use_forecast_1yr

[2015] TARGET YEAR
  Outcome masking: 17,226 rows at year >= 2015 set to NaN
  Consumption rows: 84,3

Shape validation failed: input_shape: (1, 4000), minimum_shape: (chains=2, draws=4)


  Posterior summary:
       beta0_pos: mean=0.4867  sd=0.0034  [0.4805, 0.4931]
       beta0_neg: mean=0.5445  sd=0.0224  [0.5051, 0.5865]
      beta_p_pos: mean=-0.0168  sd=0.0035  [-0.0233, -0.0104]
      beta_p_neg: mean=0.3227  sd=0.0190  [0.2878, 0.3577]
           sigma: mean=0.0700  sd=0.0004  [0.0692, 0.0708]
              nu: mean=2.0324  sd=0.0260  [1.9834, 2.0787]
  Predictions: 21,978 rows across 111 ISOs
  [h=0] 33 countries, 3267 rows | MAE=$0.813/day | RMSE=$1.742/day | MAPE=9.2% | R²=0.9777 | Cov90=88.7%
  [h=1] 27 countries, 2673 rows | MAE=$1.143/day | RMSE=$2.599/day | MAPE=9.6% | R²=0.9540 | Cov90=88.3%
  Done in 115.4s

[2016] TARGET YEAR
  Outcome masking: 16,434 rows at year >= 2016 set to NaN
  Consumption rows: 87,021, 118 ISOs, years 1977–2021, 70,587 with outcome
  GDP lookup: 4,321 entries across 120 ISOs
  dt cutoff: excluding pairs with dt >= 50 years
  Training pairs: 453 survey pairs → 44,847 obs (86 ISOs), dt range: 1–11 yrs
    GDP decomposition: 37,62

Shape validation failed: input_shape: (1, 4000), minimum_shape: (chains=2, draws=4)


  Posterior summary:
       beta0_pos: mean=0.4585  sd=0.0032  [0.4529, 0.4647]
       beta0_neg: mean=0.5164  sd=0.0201  [0.4786, 0.5525]
      beta_p_pos: mean=-0.0120  sd=0.0035  [-0.0186, -0.0056]
      beta_p_neg: mean=0.3064  sd=0.0196  [0.2705, 0.3417]
           sigma: mean=0.0679  sd=0.0005  [0.0672, 0.0688]
              nu: mean=2.0901  sd=0.0209  [2.0508, 2.1295]
  Predictions: 22,374 rows across 113 ISOs
  [h=0] 27 countries, 2673 rows | MAE=$0.892/day | RMSE=$2.150/day | MAPE=7.9% | R²=0.9618 | Cov90=89.6%
  [h=1] 24 countries, 2376 rows | MAE=$0.780/day | RMSE=$1.463/day | MAPE=7.7% | R²=0.9695 | Cov90=89.1%
  Done in 102.7s

[2017] TARGET YEAR
  Outcome masking: 15,444 rows at year >= 2017 set to NaN
  Consumption rows: 89,001, 121 ISOs, years 1977–2022, 73,557 with outcome
  GDP lookup: 4,458 entries across 121 ISOs
  dt cutoff: excluding pairs with dt >= 50 years
  Training pairs: 479 survey pairs → 47,421 obs (87 ISOs), dt range: 1–11 yrs
    GDP decomposition: 39,99

...

  Posterior summary:
       beta0_pos: mean=0.4358  sd=0.0030  [0.4304, 0.4407]
       beta0_neg: mean=0.7658  sd=0.0168  [0.7338, 0.7961]
      beta_p_pos: mean=-0.0126  sd=0.0027  [-0.0174, -0.0073]
      beta_p_neg: mean=0.0269  sd=0.0164  [-0.0011, 0.0594]
           sigma: mean=0.0644  sd=0.0004  [0.0637, 0.0651]
              nu: mean=2.2599  sd=0.0264  [2.2086, 2.3071]
  Predictions: 23,562 rows across 119 ISOs
  [h=0] 8 countries, 792 rows | MAE=$0.866/day | RMSE=$1.836/day | MAPE=6.9% | R²=0.9563 | Cov90=87.0%
  [h=1] 1 countries, 99 rows | MAE=$0.162/day | RMSE=$0.217/day | MAPE=1.8% | R²=0.9987 | Cov90=100.0%
  Done in 128.7s

[2024] TARGET YEAR
  Outcome masking: 99 rows at year >= 2024 set to NaN
  Consumption rows: 90,684, 122 ISOs, years 1977–2024, 90,585 with outcome
  GDP lookup: 5,291 entries across 122 ISOs
  dt cutoff: excluding pairs with dt >= 50 years
  Training pairs: 598 survey pairs → 59,202 obs (91 ISOs), dt range: 1–11 yrs
    GDP decomposition: 49,104 pure 

Shape validation failed: input_shape: (1, 4000), minimum_shape: (chains=2, draws=4)


  Posterior summary:
       beta0_pos: mean=0.4256  sd=0.0025  [0.4211, 0.4305]
       beta0_neg: mean=0.9157  sd=0.0185  [0.8820, 0.9512]
      beta_p_pos: mean=-0.0183  sd=0.0026  [-0.0227, -0.0134]
      beta_p_neg: mean=0.0943  sd=0.0200  [0.0578, 0.1303]
           sigma: mean=0.0642  sd=0.0004  [0.0635, 0.0648]
              nu: mean=2.2617  sd=0.0263  [2.2132, 2.3128]
  Predictions: 23,166 rows across 117 ISOs
  [h=0] 1 countries, 99 rows | MAE=$0.115/day | RMSE=$0.278/day | MAPE=0.9% | R²=0.9993 | Cov90=100.0%
  [h=1] 11583 predictions, no actuals for evaluation
  Done in 128.6s

[2025] TARGET YEAR
  Outcome masking: 0 rows at year >= 2025 set to NaN
  Consumption rows: 90,684, 122 ISOs, years 1977–2024, 90,684 with outcome
  GDP lookup: 5,437 entries across 122 ISOs
  dt cutoff: excluding pairs with dt >= 50 years
  Training pairs: 608 survey pairs → 60,192 obs (92 ISOs), dt range: 1–11 yrs
    GDP decomposition: 49,896 pure expansion, 4,158 pure contraction, 6,138 mixed


Shape validation failed: input_shape: (1, 4000), minimum_shape: (chains=2, draws=4)


  Posterior summary:
       beta0_pos: mean=0.4251  sd=0.0025  [0.4205, 0.4296]
       beta0_neg: mean=0.8586  sd=0.0174  [0.8264, 0.8899]
      beta_p_pos: mean=-0.0196  sd=0.0026  [-0.0244, -0.0146]
      beta_p_neg: mean=0.0782  sd=0.0184  [0.0442, 0.1107]
           sigma: mean=0.0641  sd=0.0004  [0.0634, 0.0648]
              nu: mean=2.2247  sd=0.0239  [2.1790, 2.2667]
  Predictions: 23,364 rows across 118 ISOs
  [h=0] 11682 predictions, no actuals for evaluation
  [h=1] 11682 predictions, no actuals for evaluation
  Done in 131.6s

WRITING FINAL MASTER OUTPUTS  (FORECAST_1YR)

Con φ v1  USE  complete.
Mode   : forecast  |  horizon: 1 yr
Outputs: /content/drive/MyDrive/conphi/outputs/conphi_v1_use_forecast_1yr
Runtime: 22.1 minutes
